[README](README.md) | [Introduction](Introduction.md) | [Datasets](Datasets.md) | Notebook

# BGP Control Plane: RPKI and Route Origin Validation

In [ ]:
name = "YOUR NAME HERE"
date = "MM/DD/YYYY"

In [ ]:
%pip install pelicanfs pybgpkit-parser pandas

In [ ]:
import pybgpkit_parser as bgpkit
import pandas as pd
from pelicanfs.core import OSDFFileSystem
from IPython.display import Markdown, display
from pathlib import Path

BEACONS = [
    ("93.175.146.0/24", "valid"),
    ("93.175.147.0/24", "invalid"),
    ("84.205.83.0/24",  "unknown"),
]
ROA_STATUS = {prefix: status for prefix, status in BEACONS}

RIB_URL = "https://osdf-director.osg-htc.org/routeviews/route-views4/bgpdata/2023.03/RIBS/rib.20230322.1800.bz2"

---

## Task 1: Explore BGP Data and RPKI Beacons

The dataset is a **full RIB (Routing Information Base) snapshot** from the RouteViews route-views4 collector,
captured at 18:00 UTC on March 22, 2023. It records every prefix that every collector peer was announcing
at that time. The file is in MRT binary format (~94 MB compressed) hosted on OSDF.

### 1.1 Browse Available RIB Files

Run the cell below to list the RIB snapshots available for March 2023 and find the one we will use.

In [ ]:
osdf = OSDFFileSystem()
objects = osdf.ls('/routeviews/route-views4/bgpdata/2023.03/RIBS')

df_files = pd.DataFrame(objects)
df_files["file"] = df_files["name"].str.split("/").str[-1]
df_files["size_mb"] = (df_files["size"] / 1e6).round(1)
df_files[df_files["file"].str.contains("20230322")][["file", "size_mb", "modified"]]

### 1.2 Explore the RPKI Beacon Prefixes

Stream the RIB snapshot directly from OSDF using bgpkit. Filter to the three RPKI beacon prefixes
and display the raw routes as a DataFrame.

Look at the `peer_asn`, `origin_asns`, and `as_path` fields.
RIPE NCC (AS 12654) operates the valid and invalid beacons. The invalid beacon
(`93.175.147.0/24`) is deliberately announced from an AS that does not match its ROA —
peers that enforce ROV (Route Origin Validation) will drop it.

In [ ]:
beacon_prefixes = set(ROA_STATUS)
rows = []
for e in bgpkit.Parser(url=RIB_URL):
    if e.prefix in beacon_prefixes:
        rows.append({
            "prefix":      e.prefix,
            "roa_status":  ROA_STATUS[e.prefix],
            "peer_asn":    e.peer_asn,
            "origin_asns": str(e.origin_asns),
            "as_path":     str(e.as_path),
        })

df_beacons = pd.DataFrame(rows)
print(f"Total beacon routes: {len(df_beacons)}")
df_beacons.head(20)

In [ ]:
# Routes per beacon prefix
df_beacons.groupby(["prefix", "roa_status"]).size().reset_index(name="route_count")

### Question 1

Approximately how many collector peers announce the ROA-valid beacon (`93.175.146.0/24`)?
How does this count compare to the ROA-invalid beacon (`93.175.147.0/24`)?
What does this difference suggest about ROV deployment?

YOUR ANSWER HERE

### Question 2

Look at the origin ASN(s) for the ROA-invalid beacon in the output above.
Is the origin AS the same as for the ROA-valid beacon? Why or why not?

YOUR ANSWER HERE

### Question 3

The ROA-invalid beacon is deliberately misconfigured by RIPE NCC.
Why does it still appear in routing tables at all?
What does its continued presence tell you about the current state of ROV enforcement on the Internet?

YOUR ANSWER HERE

---

## Task 2: RPKI Beacon Peer Summary

Using `df_beacons` from Task 1, count the distinct collector peers and origin ASNs
for each beacon prefix. Produce **Table 1** and write it to `tables/beacon-updates.md`.

### Table 1 format

| prefix | roa_status | peers_announcing | unique_origin_asns |
| --- | --- | ---: | ---: |
| 93.175.146.0/24 | valid | [count] | [count] |
| 93.175.147.0/24 | invalid | [count] | [count] |
| 84.205.83.0/24 | unknown | [count] | [count] |

**Hint:** For each row in `df_beacons`, `peer_asn` is an integer and `origin_asns` is a
string representation of a Python set (e.g. `"{12654}"`). Use `ast.literal_eval()` to parse it.

In [ ]:
import ast
from collections import defaultdict

peers_per_prefix   = defaultdict(set)
origins_per_prefix = defaultdict(set)

# YOUR CODE HERE
# Iterate over rows in df_beacons.
# For each row:
#   - add row["peer_asn"] to peers_per_prefix[row["prefix"]]
#   - parse row["origin_asns"] with ast.literal_eval() and add all values
#     to origins_per_prefix[row["prefix"]]


# Build Table 1 as a Markdown string and assign to table1.
# Columns: prefix, roa_status, peers_announcing, unique_origin_asns
# Rows: one per beacon in BEACONS order (valid, invalid, unknown)
# Right-align the count columns.

table1 = None  # replace with your Markdown string

if table1 is None:
    raise RuntimeError("Replace the None above with your Markdown table string.")

display(Markdown(table1))
Path("tables").mkdir(exist_ok=True)
Path("tables/beacon-updates.md").write_text(table1, encoding="utf-8")
print("Wrote tables/beacon-updates.md")

### Question 4

What fraction of peers that announce the valid beacon also announce the invalid one?
What fraction appear to be filtering the invalid beacon (i.e., announce valid but not invalid)?

YOUR ANSWER HERE

### Question 5

How many unique origin ASNs appear for the invalid beacon?
How many for the valid beacon?
What would it mean if the invalid beacon had multiple distinct origin ASNs?

YOUR ANSWER HERE

---

## Task 3: Measure ROV Deployment

Categorize each collector peer by its observed behavior and produce **Table 2**.

- **forwarded invalid** — peer has a route for `93.175.147.0/24` (the ROA-invalid beacon)
- **enforced ROV** — peer has a route for the valid beacon but *not* the invalid one
- **forwarded neither** — peer has no route for either beacon

### Table 2 format

| category | peers | percentage |
| --- | ---: | ---: |
| forwarded invalid | [count] | [%] |
| enforced ROV (valid, not invalid) | [count] | [%] |
| forwarded neither beacon | [count] | [%] |
| total peers | [count] | — |

In [ ]:
VALID_BEACON   = "93.175.146.0/24"
INVALID_BEACON = "93.175.147.0/24"

# YOUR CODE HERE
# Build three sets from df_beacons:
#   peers_valid   = set of peer_asn values where prefix == VALID_BEACON
#   peers_invalid = set of peer_asn values where prefix == INVALID_BEACON
#   peers_all     = union of both sets
#
# Then compute:
#   forwarded_invalid = len(peers_invalid)
#   enforced_rov      = len(peers_valid - peers_invalid)
#   forwarded_neither = len(peers_all - peers_valid - peers_invalid)
#   total_peers       = len(peers_all)
#
# Build Table 2 as a Markdown string and assign to table2.
# percentage = count / total_peers * 100, formatted to one decimal place.
# The total row uses "—" instead of a percentage.

table2 = None  # replace with your Markdown table string

if table2 is None:
    raise RuntimeError("Replace the None above with your Markdown table string.")

display(Markdown(table2))
Path("tables/rov-peers.md").write_text(table2, encoding="utf-8")
print("Wrote tables/rov-peers.md")

### Question 6

What percentage of peers in the snapshot appear to enforce ROV?
Is this surprisingly high or low given that RPKI and ROV have been technically available since around 2012?

YOUR ANSWER HERE

### Question 7

This method infers ROV from a missing route.
Name one alternative explanation for why a peer might not have a route for the invalid prefix
that has nothing to do with ROV enforcement.

YOUR ANSWER HERE

### Question 8

What additional data — not available from this single RIB snapshot — would make you
more confident in your ROV enforcement estimate?

YOUR ANSWER HERE

[README](README.md) | [Introduction](Introduction.md) | [Datasets](Datasets.md) | Notebook